In [4]:
%%capture
!pip install lightning adversarial-robustness-toolbox

In [69]:
import os
import sys
from pathlib import Path

import h5py

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from torchvision import transforms
import torchvision.models as models
from torchvision.datasets import PCAM

import torchmetrics

import numpy as np
from tqdm.notebook import tqdm

from art.attacks.evasion import FastGradientMethod
from art.estimators.classification import PyTorchClassifier

import lightning as L
from lightning import LightningModule, LightningDataModule

In [6]:
EPOCHS = 10
BATCH_SIZE = 512
SEED = 42

In [7]:
# TODO: add good transforms for training and validation datasets (roto-translation, cropping, illuminance, etc.)
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Model

In [8]:
class ResNet18Classifier(LightningModule):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.model = models.resnet18(weights='DEFAULT')
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        self.num_classes = num_classes
        
        self.criterion = nn.CrossEntropyLoss()

        self.train_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.val_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.test_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage: str):
        images, labels = batch

        outputs = self(images)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)

        if stage == "train":
            self.train_acc(preds, labels)
        elif stage == "val":
            self.val_acc(preds, labels)
        elif stage == "test":
            self.test_acc(preds, labels)
        else:
            raise ValueError(f"Unknown stage: {stage}")
        
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"{stage}_acc", getattr(self, f"{stage}_acc"), on_step=False, on_epoch=True, prog_bar=True)
        
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")
    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")
    def test_step(self, batch, batch_idx):
        return self._step(batch, "test")

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=1e-3, weight_decay=0.05)
        
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.1, patience=5
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1,
            },
        }

# Dataset

In [38]:
class PCAMDataset(torch.utils.data.Dataset):
    def __init__(self, input_file_path: str, label_file_path: str, transform=None, lazy: bool = False):
        self.input_file_path = input_file_path
        self.label_file_path = label_file_path
        self.transform = transform
        self.lazy = lazy

        input_files = h5py.File(self.input_file_path)["x"]
        self.input_files = input_files
        target_files = h5py.File(self.label_file_path)["y"]
        self.target_files = target_files

        if not lazy:
            self.images = [Image.fromarray(file).convert("RGB") for file in tqdm(input_files)]
            self.targets = [int(target_files[i, 0, 0, 0]) for i in tqdm(range(len(target_files)))]
        
    def __len__(self) -> int:
        return len(self.input_files)
            
    def __getitem__(self, idx: int):
        if not self.lazy:
            image, target = self.images[idx], torch.tensor(self.targets[idx])
        else:
            image = Image.fromarray(self.input_files[idx]).convert("RGB")
            target = int(self.target_files[idx, 0, 0, 0])
    
        if self.transform:
            image = self.transform(image)

        return image, target

In [39]:
train_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/training_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_train_y.h5",
    transform=train_transform,
    lazy=True
)

In [40]:
val_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/validation_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_valid_y.h5",
    transform=train_transform,
    lazy=True
)

In [41]:
test_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/test_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_test_y.h5",
    transform=train_transform,
    lazy=True
)

# Data Module

In [42]:
class PCAMDataModule(LightningDataModule):
    def __init__(self, train, val, test=None, name=""):
        super().__init__()
        self.train = train
        self.val = val
        self.test = test
        
        self.name = name

    def setup(self, stage: str = None):
        self.train_dataset = self.train
        self.val_dataset = self.val
        self.test_dataset = self.test

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [43]:
model = ResNet18Classifier(num_classes=2)

In [44]:
model

ResNet18Classifier(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, 

In [45]:
datamodule = PCAMDataModule(
    train=train_data,
    val=val_data,
    test=test_data
)

In [46]:
ckbs = [
    L.pytorch.callbacks.ModelCheckpoint(
            dirpath="/kaggle/working/checkpoints/",
            monitor="val_loss",
            mode="min",
            save_top_k=1,
            filename="{epoch:02d}-{val_loss:.2f}",
        ),
    L.pytorch.callbacks.LearningRateMonitor(logging_interval='epoch'),
    L.pytorch.callbacks.RichProgressBar(),
    L.pytorch.callbacks.early_stopping.EarlyStopping(
        monitor="val_loss", min_delta=0.00, patience=5, verbose=False, mode="min"
    )
]

In [47]:
tensorboard_logger = L.pytorch.loggers.TensorBoardLogger(
    save_dir="/kaggle/working/logs/",
)

In [48]:
L.seed_everything(SEED, workers=True)

trainer = L.Trainer(
    accelerator="auto", 
    devices="auto",
    max_epochs=EPOCHS,
    precision="16-mixed", # Use mixed precision for faster training
    num_nodes=1,
    logger=[tensorboard_logger],
    callbacks=ckbs,
    log_every_n_steps=10,
)

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs


In [49]:
trainer.fit(model, datamodule=datamodule)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ ResNet           │ 11.2 M │ train │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │
│ 2 │ train_acc │ BinaryAccuracy   │      0 │ train │
│ 3 │ val_acc   │ BinaryAccuracy   │      0 │ train │
│ 4 │ test_acc  │ BinaryAccuracy   │      0 │ train │
└───┴───────────┴──────────────────┴────────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 72                                                                                          
Modules in eval mode: 0

Output()

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [50]:
trainer.test(model, datamodule=datamodule, ckpt_path="best")

INFO: Restoring states from the checkpoint path at /kaggle/working/checkpoints/epoch=00-val_loss=0.30.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at /kaggle/working/checkpoints/epoch=00-val_loss=0.30.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │       0.8427734375        │
│         test_loss         │    0.44868674874305725    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.44868674874305725, 'test_acc': 0.8427734375}]

In [ ]:
classifier = PyTorchClassifier(
    model=model.model,
    loss=nn.CrossEntropyLoss(),
    optimizer=optim.AdamW(model.model.parameters(), lr=1e-3, weight_decay=0.05),
    input_shape=(3, 96, 96),
    nb_classes=2,
)

x_test = train_data[0][0].unsqueeze(0).cpu().numpy()
y_test = torch.tensor(train_data[0][1]).unsqueeze(0).cpu().numpy()

x_test.shape, y_test

np.sum(np.argmax(classifier.predict(x_test), axis=1) == y_test) / len(y_test)

# Step 6: Generate adversarial test examples
attack = FastGradientMethod(estimator=classifier, eps=0.2)
x_test_adv = attack.generate(x=x_test)

# Step 7: Evaluate the ART classifier on adversarial test examples

predictions = classifier.predict(x_test_adv)
accuracy = np.sum(np.argmax(predictions, axis=1) == y_test) / len(y_test)
print("Accuracy on adversarial test examples: {}%".format(accuracy * 100))

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

def denormalize(tensor, mean, std):
    """
    Denormalizes a tensor image using the provided mean and std.
    """
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return tensor * std + mean

import matplotlib.pyplot as plt
import torch
import numpy as np

# Convert NumPy arrays to PyTorch tensors
original = torch.tensor(x_test[0])       # Shape: (3, 96, 96)
adversarial = torch.tensor(x_test_adv[0])  # Shape: (3, 96, 96)

# Denormalize the images
original_denorm = denormalize(original, mean, std)
adversarial_denorm = denormalize(adversarial, mean, std)

# Clip the values to [0,1] range for display
original_denorm = torch.clamp(original_denorm, 0, 1)
adversarial_denorm = torch.clamp(adversarial_denorm, 0, 1)

# Convert tensors to NumPy arrays and transpose to HWC for matplotlib
original_img = original_denorm.permute(1, 2, 0).numpy()
adversarial_img = adversarial_denorm.permute(1, 2, 0).numpy()

# Plot the images
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(original_img)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Adversarial Image")
plt.imshow(adversarial_img)
plt.axis('off')

plt.tight_layout()
plt.show()

In [161]:
adversarial_test_dataset = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/test_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_test_y.h5",
    lazy=False
)

  0%|          | 0/32768 [00:00<?, ?it/s]

  0%|          | 0/32768 [00:00<?, ?it/s]

In [163]:
N = len(adversarial_test_dataset)

# (A) Pre-allocate two lists to hold each sample after transform:
all_imgs = []
all_labels = []

for idx in tqdm(range(N)):
    img_pil = adversarial_test_dataset.images[idx]         # a PIL Image (RGB)
    lbl_int = adversarial_test_dataset.targets[idx]         # an integer 0/1

    # (B) Apply your transform (ToTensor + Normalize) to produce a Tensor(3×96×96):
    img_tensor = train_transform(img_pil)   # shape: (3, 96, 96)

    all_imgs.append(img_tensor.unsqueeze(0))       # shape becomes (1, 3, 96, 96)
    all_labels.append(torch.tensor([lbl_int]))     # shape (1,)

# (C) Concatenate into two big tensors:
x_all = torch.cat(all_imgs, dim=0)    # shape: (N, 3, 96, 96)
y_all = torch.cat(all_labels, dim=0)  # shape: (N,)

# (D) Convert to NumPy arrays for ART:
x_test = x_all.cpu().numpy()          # dtype float32, still normalized
y_test = y_all.cpu().numpy().astype(int)

  0%|          | 0/32768 [00:00<?, ?it/s]

In [ ]:
eps_list = np.linspace(0.0, 0.3, num=10)
accuracies = []

for eps in tqdm(eps_list):
    attack = FastGradientMethod(estimator=classifier, eps=eps)
    x_adv = attack.generate(x=x_test)            # still shape (N, 3, 96, 96)
    probs_adv = classifier.predict(x_adv)         # shape (N, 2)
    y_adv_pred = np.argmax(probs_adv, axis=1)
    acc_adv = (y_adv_pred == y_test).mean()
    accuracies.append(acc_adv)

# Plot
plt.figure(figsize=(7, 5))
plt.plot(eps_list, accuracies, marker='o', linestyle='-')
plt.title("Accuracy vs. FGSM ε")
plt.xlabel("ε")
plt.ylabel("Accuracy on Adversarial Examples")
plt.grid(True)
plt.ylim(0, 1.05)
plt.xticks(eps_list)
plt.show()

  0%|          | 0/10 [00:00<?, ?it/s]